## Downloading USGS stream flow and water level data

In [1]:
import datetime as dt
import os
import polars as pl
import pandas as pd
from dataretrieval import waterdata # Python wrapper for USGS' Water Data APIs

from utils import display_cols, DATE_FORMAT

Geopandas not installed. Geometries will be flattened into pandas DataFrames.


In [2]:
API_USGS_PAT = os.getenv('API_KEY_USGS')

if API_USGS_PAT:
    os.environ['API_KEY_USGS'] = API_USGS_PAT
else:
    print("No USGS API key detected in .env under API_KEY_USGS.")

No USGS API key detected in .env under API_KEY_USGS.


In [ ]:
FPATH_DATA = '../data/'
FPATH_DATA_STATIONS = FPATH_DATA + 'stations.csv'


STATISTIC_ID_MEAN = '00003'
STATISTIC_ID_UTC2400 = '32400'
STATISTIC_ID_INSTANTANEOUS = '00011'
STATISTIC_IDS = [STATISTIC_ID_MEAN, STATISTIC_ID_UTC2400] # utc2400 needed for old FH Lake data
PARAMETER_CODE_DISCHARGE = '00060'
# PARAMETER_CODE_GAGEHEIGHT = '00065' # Staging variable to compute discharge, not needed
PARAMETER_CODE_RESERVOIRELEV = '00062'
PARAMETER_CODE_ELEVATION = '62614'
PARAMETER_CODE_ELEVATIONALT = '62615'
PARAMETER_CODES = [PARAMETER_CODE_DISCHARGE, PARAMETER_CODE_RESERVOIRELEV, PARAMETER_CODE_ELEVATION]

TIME_EARLIEST = '1910-10-01'
TIME_LATEST = ''
TIME_FULL = '2024-03-01/2024-03-05'

In [ ]:
dt.datetime.today().date().strftime(DATE_FORMAT)

'2026-07-08'

### Load monitoring stations of interest

In [5]:
stations_csv = pl.read_csv(FPATH_DATA_STATIONS)

In [6]:
# 'is_disjoint' denotes monitoring stations of interest for this analysis
station_ids = stations_csv.filter(pl.col('is_disjoint') == True)['station_id'].to_list()
station_ids

['USGS-12370000',
 'USGS-12371550',
 'USGS-12372000',
 'USGS-12365700',
 'USGS-12366000',
 'USGS-12362500',
 'USGS-12358500',
 'USGS-12355500']

In [7]:
stations_usgs = waterdata.get_monitoring_locations(
    monitoring_location_id=station_ids,
    
)

Retrieving: monitoring-locations · 1 page · 8 rows · 996/1,000 requests remaining


In [8]:
# Find which data are measured at each station
time_series_usgs = waterdata.get_time_series_metadata(
    monitoring_location_id=station_ids,
    skip_geometry=True
)[0]

Retrieving: time-series-metadata · 1 page · 61 rows · 996/1,000 requests remaining


In [9]:
time_series_usgs.columns

Index(['time_series_id', 'unit_of_measure', 'parameter_name', 'parameter_code',
       'statistic_id', 'hydrologic_unit_code', 'state_name', 'last_modified',
       'begin', 'end', 'begin_utc', 'end_utc', 'computation_period_identifier',
       'computation_identifier', 'thresholds', 'sublocation_identifier',
       'primary', 'monitoring_location_id', 'web_description',
       'parameter_description', 'parent_time_series_id', 'data_gap_interval'],
      dtype='str')

In [10]:
# What are all of the data types (means) observed at our stations? Water temp, discharge, etc.
time_series_usgs.loc[
    time_series_usgs['statistic_id']==STATISTIC_ID_MEAN, 
    ['parameter_name', 'parameter_code', 'parameter_description', 'unit_of_measure']
].drop_duplicates() # monitoring_location_id
# Gage height: Water level relative to local reference point (staging value, not used for lake level)
# Elevation, lake/res, NGVD29: Water level relative to an absolute reference point
# Reservoir height: Also an absolute measure

,parameter_name,parameter_code,parameter_description,unit_of_measure
10,"Temperature, water",00010,"Temperature, water, degrees Celsius",degC
11,Suspnd sedmnt disch,80155,"Suspended sediment discharge, short tons per day",tons/day
19,Discharge,00060,"Discharge, cubic feet per second",ft^3/s
30,Reservoir elevation,00062,Elevation of reservoir water surface above dat...,ft
45,Suspnd sedmnt conc,80154,"Suspended sediment concentration, milligrams p...",mg/l


In [ ]:
display_cols(time_series_usgs.loc[
        time_series_usgs['monitoring_location_id'] == 'USGS-12371550',
        :
    ])

,time_series_id,unit_of_measure,parameter_name,parameter_code,statistic_id,hydrologic_unit_code,state_name,last_modified,begin,end,begin_utc,end_utc,computation_period_identifier,computation_identifier,thresholds,sublocation_identifier,primary,monitoring_location_id,web_description,parameter_description,parent_time_series_id,data_gap_interval
30,7b86eea3950548f3afd281b06f4e73cb,ft,Reservoir elevation,00062,00003,170102080404,Montana,2026-07-08 07:37:51.012962,1999-10-01 00:00:00.000001,2026-07-06 00:00:00.000001,1999-10-01 06:00:00+00:00,2026-07-06 06:00:00+00:00,Daily,Mean,[],None,Primary,USGS-12371550,Somers datum,Elevation of reservoir water surface above dat...,a3990f8fe32a42e9a13522748b385e4f,NaN
41,a3990f8fe32a42e9a13522748b385e4f,ft,Reservoir elevation,00062,00011,170102080404,Montana,2026-07-08 06:37:17.882283,2007-10-01 07:00:00.000001,2026-07-08 06:30:00.000001,2007-10-01 13:00:00+00:00,2026-07-08 12:30:00+00:00,Points,Instantaneous,[{'Name': 'Lower Staff elevation need to run l...,None,Primary,USGS-12371550,Somers datum,Elevation of reservoir water surface above dat...,NaN,PT1H12M
53,dc771b139c61433394e5da4e06b526d3,ft,Reservoir elevation,00062,32400,170102080404,Montana,2017-03-06 06:15:23.404011,1909-04-06 00:00:00.000001,1999-09-30 00:00:00.000001,1909-04-06 07:00:00+00:00,1999-09-30 06:00:00+00:00,Daily,SelectedValue,[],None,Primary,USGS-12371550,NaN,Elevation of reservoir water surface above dat...,NaN,NaN


In [28]:
# Find which values (discharge, gage height) and for which dates we have MEAN data at each station
time_series_available = time_series_usgs.loc[
    time_series_usgs['parameter_code'].isin(PARAMETER_CODES) & time_series_usgs['statistic_id'].isin(STATISTIC_IDS), 
    [
        'monitoring_location_id', 
        'time_series_id', 
        'parameter_name', 
        'parameter_code', 
        'statistic_id', 
        'unit_of_measure', 
        'begin', 
        'end'
    ]
]

In [29]:
stations_csv_pd = pd.read_csv(FPATH_DATA_STATIONS)
# stations_csv_pd.rename(columns={'station_id':'monitoring_location_id'}, inplace=True)
# stations_csv.select(['station_name', 'station_shorthand', 'station_id'])

In [30]:
time_series_available = time_series_available.merge(
    stations_csv_pd.loc[:, ['station_name', 'station_shorthand', 'station_id']],
    left_on = 'monitoring_location_id',
    right_on = 'station_id'
)

In [31]:
time_series_available.shape

(9, 11)

In [32]:
len(station_ids)

8

In [33]:
import itertools

In [34]:
# Exclude HH Reservoir for now bc it uses a separate elevation parameter (62614)
location_ids_lake = stations_csv_pd.loc[
    stations_csv_pd['site_type'].isin(['lake']) & stations_csv_pd['is_disjoint'],
    'station_id'
    ].to_list()

In [35]:
location_pram_dict = dict(zip(station_ids, itertools.repeat(PARAMETER_CODE_DISCHARGE)))


In [36]:
location_pram_dict = {
    k: (PARAMETER_CODE_RESERVOIRELEV if k in location_ids_lake else v) for k, v in location_pram_dict.items()
}

In [37]:
location_pram_df = pd.DataFrame({
    'monitoring_location_id': location_pram_dict.keys(),
    'parameter_code': location_pram_dict.values()
    })
location_pram_df

,monitoring_location_id,parameter_code
0,USGS-12370000,00060
1,USGS-12371550,00062
2,USGS-12372000,00060
3,USGS-12365700,00060
4,USGS-12366000,00060
5,USGS-12362500,00060
6,USGS-12358500,00060
7,USGS-12355500,00060


Here are all discharge and reservoir height time series available at each of our stations of interest.

In [38]:
time_series_available.sort_values(by = ['monitoring_location_id'])

,monitoring_location_id,time_series_id,parameter_name,parameter_code,statistic_id,unit_of_measure,begin,end,station_name,station_shorthand,station_id
1,USGS-12355500,7044c60248554783823aabb670451c24,Discharge,00060,00003,ft^3/s,1910-10-01 00:00:00.000001,2026-07-06 00:00:00.000001,N F Flathead River nr Columbia Falls MT,fhr_n,USGS-12355500
0,USGS-12358500,587e53501ee04585bd078dfafed6b54b,Discharge,00060,00003,ft^3/s,1939-10-01 00:00:00.000001,2026-07-06 00:00:00.000001,M F Flathead River near West Glacier MT,fhr_m,USGS-12358500
5,USGS-12362500,807167eeb3ec416e85b4cd277a41670a,Discharge,00060,00003,ft^3/s,1911-02-01 00:00:00.000001,2026-07-06 00:00:00.000001,S F Flathead River nr Columbia Falls MT,fhr_s,USGS-12362500
6,USGS-12365700,9b9c37eb614d4a6d9fcb938befc5ca07,Discharge,00060,00003,ft^3/s,2007-03-01 00:00:00.000001,2026-07-06 00:00:00.000001,"Stillwater River at Lawrence Park, at Kalispell",stillwater,USGS-12365700
7,USGS-12366000,bbeb6445470d427f84aa6ec49e7c66b1,Discharge,00060,00003,ft^3/s,1928-08-01 00:00:00.000001,2026-07-06 00:00:00.000001,Whitefish River near Kalispell MT,whitefish,USGS-12366000
2,USGS-12370000,7b04109277f74a648002d475466fec30,Discharge,00060,00003,ft^3/s,1922-05-01 00:00:00.000001,2026-07-06 00:00:00.000001,"Swan River near Bigfork, MT",swan,USGS-12370000
3,USGS-12371550,7b86eea3950548f3afd281b06f4e73cb,Reservoir elevation,00062,00003,ft,1999-10-01 00:00:00.000001,2026-07-06 00:00:00.000001,Flathead Lake at Polson MT,fhl,USGS-12371550
8,USGS-12371550,dc771b139c61433394e5da4e06b526d3,Reservoir elevation,00062,32400,ft,1909-04-06 00:00:00.000001,1999-09-30 00:00:00.000001,Flathead Lake at Polson MT,fhl,USGS-12371550
4,USGS-12372000,7db63fb262c845f7904cd34285cc4c21,Discharge,00060,00003,ft^3/s,1907-08-01 00:00:00.000001,2026-07-06 00:00:00.000001,Flathead River near Polson MT,fhr_out,USGS-12372000


In [39]:
# Why is gage height only ever reported with 00011, instantaneous statistic values? No means?
# Answer: Gage height is a "staging" value, meaning those are the values used to compute discharge means!

Now, we filter down to just the time series that we want, those being daily mean discharge for rivers and daily mean reservoir height for "lakes".

In [41]:
time_series_available.merge(location_pram_df, on=['monitoring_location_id', 'parameter_code'])

,monitoring_location_id,time_series_id,parameter_name,parameter_code,statistic_id,unit_of_measure,begin,end,station_name,station_shorthand,station_id
0,USGS-12358500,587e53501ee04585bd078dfafed6b54b,Discharge,00060,00003,ft^3/s,1939-10-01 00:00:00.000001,2026-07-06 00:00:00.000001,M F Flathead River near West Glacier MT,fhr_m,USGS-12358500
1,USGS-12355500,7044c60248554783823aabb670451c24,Discharge,00060,00003,ft^3/s,1910-10-01 00:00:00.000001,2026-07-06 00:00:00.000001,N F Flathead River nr Columbia Falls MT,fhr_n,USGS-12355500
2,USGS-12370000,7b04109277f74a648002d475466fec30,Discharge,00060,00003,ft^3/s,1922-05-01 00:00:00.000001,2026-07-06 00:00:00.000001,"Swan River near Bigfork, MT",swan,USGS-12370000
3,USGS-12371550,7b86eea3950548f3afd281b06f4e73cb,Reservoir elevation,00062,00003,ft,1999-10-01 00:00:00.000001,2026-07-06 00:00:00.000001,Flathead Lake at Polson MT,fhl,USGS-12371550
4,USGS-12372000,7db63fb262c845f7904cd34285cc4c21,Discharge,00060,00003,ft^3/s,1907-08-01 00:00:00.000001,2026-07-06 00:00:00.000001,Flathead River near Polson MT,fhr_out,USGS-12372000
5,USGS-12362500,807167eeb3ec416e85b4cd277a41670a,Discharge,00060,00003,ft^3/s,1911-02-01 00:00:00.000001,2026-07-06 00:00:00.000001,S F Flathead River nr Columbia Falls MT,fhr_s,USGS-12362500
6,USGS-12365700,9b9c37eb614d4a6d9fcb938befc5ca07,Discharge,00060,00003,ft^3/s,2007-03-01 00:00:00.000001,2026-07-06 00:00:00.000001,"Stillwater River at Lawrence Park, at Kalispell",stillwater,USGS-12365700
7,USGS-12366000,bbeb6445470d427f84aa6ec49e7c66b1,Discharge,00060,00003,ft^3/s,1928-08-01 00:00:00.000001,2026-07-06 00:00:00.000001,Whitefish River near Kalispell MT,whitefish,USGS-12366000
8,USGS-12371550,dc771b139c61433394e5da4e06b526d3,Reservoir elevation,00062,32400,ft,1909-04-06 00:00:00.000001,1999-09-30 00:00:00.000001,Flathead Lake at Polson MT,fhl,USGS-12371550


### Water time series data

For ease of comparison, we will use the time series metadata to find the latest start dates and earliest end dates among all series. That way, we have a window of time with daily values at every monitoring location.

First, we have to treat the 2 time series found above for Flathead Lake's water level. Before 1999, the reservoir elevation was measured at UTC 2400, not a daily mean like all the rest of our data. Luckily, the datum is the same (Somers), and a single 5:00PM MST reading each day is good enough for this analysis. We can concatenate the 2 Flathead Lake time series.

#### Discharge
We manually had to verify why the lake was split into 2 time series, so now we know we have over a century of data. Let's programmatically check the date range for which we have data on *all* streams.

In [47]:
station_ids_stream = stations_csv_pd.loc[
    (stations_csv_pd['site_type']=='stream') & (stations_csv_pd['is_disjoint']),
    'station_id'
].to_list()
station_ids_stream

['USGS-12370000',
 'USGS-12372000',
 'USGS-12365700',
 'USGS-12366000',
 'USGS-12362500',
 'USGS-12358500',
 'USGS-12355500']

In [52]:
date_discharge_dates = time_series_available.loc[
    time_series_available['monitoring_location_id'].isin(station_ids_stream),
    :
].agg({
    'begin': ['max'],
    'end': ['min']
})

In [ ]:
date_discharge_earliest = date_discharge_dates.loc['max', 'begin'].strftime(DATE_FORMAT)
date_discharge_latest = date_discharge_dates.loc['min', 'end'].strftime(DATE_FORMAT)
date_discharge_dates

,begin,end
max,2007-03-01 00:00:00.000001,NaT
min,NaT,2026-07-06 00:00:00.000001


In [61]:
date_discharge_earliest

'2007-03-01'

Stillwater is a bottleneck! There is only data on the Stillwater as far back as 2007.

In [75]:
discharge_raw = waterdata.get_daily(
    monitoring_location_id=station_ids_stream,
    parameter_code=PARAMETER_CODE_DISCHARGE,
    statistic_id=STATISTIC_ID_MEAN,
    time=date_discharge_earliest + '/' + date_discharge_latest,
    skip_geometry=True
)

Retrieving: daily · 1 page · 44,861 rows · 998/1,000 requests remaining


In [76]:
discharge_raw[0]

,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,daily_id
0,7044c60248554783823aabb670451c24,USGS-12355500,00060,00003,2007-03-01,668.0,ft^3/s,Approved,None,2025-03-05 03:36:25.968979+00:00,dc44ec20-f374-4097-9b93-a7b14eacc8be
1,587e53501ee04585bd078dfafed6b54b,USGS-12358500,00060,00003,2007-03-01,550.0,ft^3/s,Approved,None,2025-02-24 14:24:12.334135+00:00,e1dc3bf7-3120-4b3b-81bd-c59f9a896d90
2,807167eeb3ec416e85b4cd277a41670a,USGS-12362500,00060,00003,2007-03-01,2420.0,ft^3/s,Approved,None,2025-03-11 09:16:59.953601+00:00,0517e13a-b436-4848-a24c-f243063dca12
3,9b9c37eb614d4a6d9fcb938befc5ca07,USGS-12365700,00060,00003,2007-03-01,110.0,ft^3/s,Approved,[ESTIMATED],2025-03-11 09:19:19.316614+00:00,cc52e77a-8a5a-4f06-883a-ceb424eaa920
4,7b04109277f74a648002d475466fec30,USGS-12370000,00060,00003,2007-03-01,450.0,ft^3/s,Approved,None,2025-03-11 09:16:34.209086+00:00,d612f1e0-49f3-4afd-850a-f162bcdf93b9
...,...,...,...,...,...,...,...,...,...,...,...
44856,807167eeb3ec416e85b4cd277a41670a,USGS-12362500,00060,00003,2026-07-06,3480.0,ft^3/s,Provisional,None,2026-07-07 09:06:56.888461+00:00,66d5c64a-541e-4e6a-b55d-914b11c74944
44857,9b9c37eb614d4a6d9fcb938befc5ca07,USGS-12365700,00060,00003,2026-07-06,367.0,ft^3/s,Provisional,None,2026-07-07 09:07:30.056022+00:00,eb49ec33-cbf7-4f9b-9e2c-2032a80cbf43
44858,bbeb6445470d427f84aa6ec49e7c66b1,USGS-12366000,00060,00003,2026-07-06,293.0,ft^3/s,Provisional,None,2026-07-07 09:07:57.020175+00:00,701500cd-01b0-461b-8b1e-ac741ca53ac9
44859,7b04109277f74a648002d475466fec30,USGS-12370000,00060,00003,2026-07-06,2180.0,ft^3/s,Provisional,None,2026-07-07 09:06:52.355633+00:00,35c1fc8b-89a0-47c9-a24d-697ce76ce921


In [77]:
discharge = discharge_raw[0].merge(
    stations_csv_pd,
    left_on='monitoring_location_id',
    right_on='station_id'
)

In [95]:
discharge.head()

,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,...,where_stream,topology,site_type,pfafstetter_code,downstream_station_id,location_rel_to_dam,relevant_dam,has_flow_data,is_disjoint,notes
0,7044c60248554783823aabb670451c24,USGS-12355500,00060,00003,2007-03-01,668.0,ft^3/s,Approved,None,2025-03-05 03:36:25.968979+00:00,...,upstream,nf_fh,stream,91,USGS-12363000,NaN,NaN,True,True,NaN
1,587e53501ee04585bd078dfafed6b54b,USGS-12358500,00060,00003,2007-03-01,550.0,ft^3/s,Approved,None,2025-02-24 14:24:12.334135+00:00,...,upstream,mf_fh,stream,8,USGS-12363000,NaN,NaN,True,True,NaN
2,807167eeb3ec416e85b4cd277a41670a,USGS-12362500,00060,00003,2007-03-01,2420.0,ft^3/s,Approved,None,2025-03-11 09:16:59.953601+00:00,...,upstream,sf_fh,stream,61,USGS-12363000,downstream,Hungry Horse Dam,True,True,NaN
3,9b9c37eb614d4a6d9fcb938befc5ca07,USGS-12365700,00060,00003,2007-03-01,110.0,ft^3/s,Approved,[ESTIMATED],2025-03-11 09:19:19.316614+00:00,...,upstream,st_upr,stream,41,USGS-12371550,NaN,NaN,True,True,There is no longer a station between Flathead ...
4,7b04109277f74a648002d475466fec30,USGS-12370000,00060,00003,2007-03-01,450.0,ft^3/s,Approved,None,2025-03-11 09:16:34.209086+00:00,...,upstream,sw_upr,stream,2,USGS-12371550,upstream,Bigfork Hydroelectric Project,True,True,Before Bigfork dam; right after Swan Lake


In [90]:
discharge_date_min = discharge['time'].min().strftime(DATE_FORMAT)
discharge_date_max = discharge['time'].max().strftime(DATE_FORMAT)

In [92]:
discharge.to_csv(f'{FPATH_DATA}discharge/discharge_{discharge_date_min}_{discharge_date_max}.csv', index=None)

#### Reservoir elevation
Remember: We will be combining the modern daily mean observations with the older evening-time daily observations, if necessary to overlap elevation data with the discharge data above.

In [93]:
station_ids_lake = stations_csv_pd.loc[
    (stations_csv_pd['site_type']=='lake') & (stations_csv_pd['is_disjoint']),
    'station_id'
]
station_ids_lake

1    USGS-12371550
Name: station_id, dtype: str

In [94]:
elevation_raw_means = waterdata.get_daily(
    monitoring_location_id=station_ids_lake,
    parameter_code=PARAMETER_CODE_RESERVOIRELEV,
    statistic_id=STATISTIC_ID_MEAN,
    time=date_discharge_earliest + '/' + date_discharge_latest,
    skip_geometry=True
)

Retrieving: daily · 1 page · 7,065 rows · 999/1,000 requests remaining


In [96]:
elevation_means = elevation_raw_means[0].merge(
    stations_csv_pd,
    left_on='monitoring_location_id',
    right_on='station_id'
)

In [110]:
display_cols(elevation_means)

,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,daily_id,station_name,station_id,station_shorthand,where_stream,topology,site_type,pfafstetter_code,downstream_station_id,location_rel_to_dam,relevant_dam,has_flow_data,is_disjoint,notes
0,7b86eea3950548f3afd281b06f4e73cb,USGS-12371550,00062,00003,2007-03-01,2885.02,ft,Approved,None,2025-03-11 09:16:35.303963+00:00,81ebb0d6-fc93-4ced-a993-2fce8f8aca46,Flathead Lake at Polson MT,USGS-12371550,fhl,midstream,fh_lwr,lake,3,USGS-12372000,midstream,ALL,False,True,South end of lake
1,7b86eea3950548f3afd281b06f4e73cb,USGS-12371550,00062,00003,2007-03-02,2884.97,ft,Approved,None,2025-03-11 09:16:35.303963+00:00,c599fb0d-4798-45f0-a491-cb642ad79818,Flathead Lake at Polson MT,USGS-12371550,fhl,midstream,fh_lwr,lake,3,USGS-12372000,midstream,ALL,False,True,South end of lake
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7063,7b86eea3950548f3afd281b06f4e73cb,USGS-12371550,00062,00003,2026-07-05,2893.11,ft,Provisional,None,2026-07-06 09:07:16.335339+00:00,15b1aabf-89b5-40d5-ac15-17ca9d6b8941,Flathead Lake at Polson MT,USGS-12371550,fhl,midstream,fh_lwr,lake,3,USGS-12372000,midstream,ALL,False,True,South end of lake
7064,7b86eea3950548f3afd281b06f4e73cb,USGS-12371550,00062,00003,2026-07-06,2893.12,ft,Provisional,None,2026-07-07 09:06:53.128643+00:00,49196924-de7a-4ed1-8a1c-2edea1b6f674,Flathead Lake at Polson MT,USGS-12371550,fhl,midstream,fh_lwr,lake,3,USGS-12372000,midstream,ALL,False,True,South end of lake


In [111]:
elevation_means.to_csv(f'{FPATH_DATA}elevation/elevation{discharge_date_min}_{discharge_date_max}.csv', index=None)